# 04 — Apache Arrow and Columnar Memory

### Why this exists

Columnar memory changes how analytical data is represented and moved. This matters when tables become larger and pipelines exchange data between tools.

### Learning objectives

- explain row vs columnar layout
- inspect a columnar representation
- identify a workload that benefits

### Prerequisite

03 — CSV vs JSON vs Parquet

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C01 — System Foundations  
**Workstream:** Foundation  
**Primary roles:** Data Engineer|Analytics Engineer  
**You will prove:** Explain the mechanism and reproduce it with a deterministic experiment.


## 🖊️ Sketch note

![Sketch note — Apache Arrow & Columnar Memory](../assets/sketch-notes/04-apache-arrow-columnar-memory.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S02 — API schema drift (DATA-1051)** → [`../work_simulations/02_acme_commerce_api_schema_drift/README.md`](../work_simulations/02_acme_commerce_api_schema_drift/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

Write down one thing you expect to observe and why. Do not use the output as your prediction.


## Production bridge

Ask: what happens when the data is 100× larger, arrives late, changes shape, or is queried by several teams? State the assumption that makes your experiment useful.


## Independent challenge

Change the column types and observe how the representation changes.

### Evidence to keep

- your prediction
- one measurement
- the failure or edge case you tested
- one production implication


# 04 — Apache Arrow and Columnar Memory

## The minimum theory

Row-oriented data keeps fields for a record together. Columnar data keeps values from the same field together.

Analytical workloads often scan many rows but only a few columns, so columnar representation can reduce unnecessary work.

**Apache Arrow** is a typed, in-memory columnar format and interchange ecosystem. Think of it as a common representation that lets data tools exchange tables without repeatedly converting everything into generic Python objects.

The practical questions are: what is the schema, what columns do we need, and how many conversions happen in the pipeline?


In [ ]:
import sys,subprocess,importlib.util
if importlib.util.find_spec("pyarrow") is None: subprocess.check_call([sys.executable,"-m","pip","install","-q","pyarrow>=16,<22"])
import numpy as np,pandas as pd,pyarrow as pa,pyarrow.compute as pc,time
rng=np.random.default_rng(42)
df=pd.DataFrame({"id":np.arange(200_000,dtype=np.int64),"category":rng.choice(["A","B","C","D"],200_000),"value":rng.normal(100,20,200_000)})
table=pa.Table.from_pandas(df,preserve_index=False)
print(table.schema)


## Inspect the columnar representation

Arrow makes types explicit. A reader can access a column without reconstructing unrelated columns.

This is the foundation for ideas such as **column pruning** and efficient vectorized operations.


In [ ]:
print(table["value"].type)
print(table["value"].slice(0,5))
t=time.perf_counter(); pandas_mean=df["value"].mean(); pandas_time=time.perf_counter()-t
t=time.perf_counter(); arrow_mean=pc.mean(table["value"]).as_py(); arrow_time=time.perf_counter()-t
pd.DataFrame({"system":["pandas","Arrow compute"],"mean":[pandas_mean,arrow_mean],"seconds":[pandas_time,arrow_time]})


## Break it

Try more rows, extra string columns, and repeated `pandas → Arrow → pandas` conversion.

**Production lesson:** efficient representation does not make conversion free. Measure the whole pipeline.

**Next:** SQL fundamentals.


## Break — conversion is part of the workload

In [ ]:
# Representation speed is not the same as end-to-end pipeline speed.
t0=time.perf_counter(); roundtrip=pa.Table.from_pandas(df,preserve_index=False).to_pandas(); elapsed=time.perf_counter()-t0
print("roundtrip rows:",len(roundtrip),"seconds:",round(elapsed,6))
assert len(roundtrip)==len(df)